# NeuroShield — Hybrid CNN-LSTM-Attention Training Notebook
**Author:** Sukhman Singh  
**Organization:** C-DAC Mohali (AI & Cybersecurity Summer Training)  
**GitHub:** [sukhman-shergill/NeuroShield](https://github.com/sukhman-shergill/NeuroShield)

---

This notebook trains the **Hybrid CNN-BiLSTM-Attention** network intrusion detection model on the **UNSW-NB15** dataset using **Google Colab** (free T4 GPU). Once training completes, the final cell auto-downloads all model artifacts to your computer. Place them in the local project's `models/` and `outputs/` directories.

### ⚠️ IMPORTANT: ALWAYS UPLOAD THE LATEST NOTEBOOK FILE TO COLAB
Do **not** run from your existing browser tab. Download this file from your local computer and upload it to Colab fresh — overwriting the old version.

### UNSW-NB15 Dataset Setup
1. Download from [Kaggle](https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15) or [UNSW](https://research.unsw.edu.au/projects/unsw-nb15-dataset)
2. Upload **both** CSV files to Colab (left sidebar → Files icon, or run the Kaggle cell below):
   - `UNSW_NB15_training-set.csv`
   - `UNSW_NB15_testing-set.csv`

### Key Engineering Decisions
1. **Class-Weighted Focal Loss (γ=2.0, square-root alpha)** — Per-class alpha computed dynamically via square-root frequency scaling (Cui et al., CVPR 2019). Boosts U2R (~5.4×) and DoS (~1.7×) gradients without destabilizing majority-class training.
2. **Label Smoothing (ε=0.05)** — Soft targets prevent the softmax from collapsing to near-100% confidence on Normal/Probe, improving minority-class recall and calibration.
3. **MacroF1Score** as primary metric & EarlyStopping monitor — equal weight to all 5 classes.
4. **Chronological sequence construction + Stratified sequence splitting** — preserves packet progression context within windows; stratified split ensures both train/val see all classes.
5. **StandardScaler fit on training set only** — prevents data leakage.
6. **Tuned hyperparameters**: `BATCH_SIZE=256` (smoother minority gradients), `LR=0.001` (faster warm-start), `EARLY_STOPPING_PATIENCE=15`, `REDUCE_LR_PATIENCE=6` (60 max epochs).

See `RESEARCH_NOTES.md` for the full architecture decision record.

## 1. Install Dependencies & Configure Environment

In [ ]:
!pip install -q scikit-learn matplotlib seaborn joblib

import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print("Environment ready.")

## 2. Configuration & Hyperparameters

In [ ]:
import os

# ─── Directories ──────────────────────────────────────────────────────────────
DATA_DIR           = "./data"
RAW_DATA_DIR       = os.path.join(DATA_DIR, "raw")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")
MODELS_DIR         = "./models"
OUTPUTS_DIR        = "./outputs"
for d in [RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ─── Dataset files ────────────────────────────────────────────────────────────
TRAIN_FILE = os.path.join(RAW_DATA_DIR, "UNSW_NB15_training-set.csv")
TEST_FILE  = os.path.join(RAW_DATA_DIR, "UNSW_NB15_testing-set.csv")

CATEGORICAL_FEATURES = ["proto", "service", "state"]
SKEWED_COLS = ["dur", "sbytes", "dbytes", "rate", "sload", "dload", "smean", "dmean", "response_body_len"]

ATTACK_MAPPING = {
    "Normal":        "Normal",
    "DoS":           "DoS",
    "Fuzzers":       "Probe",
    "Reconnaissance":"Probe",
    "Analysis":      "Probe",
    "Generic":       "Probe",
    "Exploits":      "R2L",
    "Backdoor":      "R2L",
    "Backdoors":     "R2L",
    "Shellcode":     "U2R",
    "Worm":          "U2R",
    "Worms":         "U2R",
}

# ─── Class labels (alphabetical order to match LabelEncoder) ──────────────────
CLASS_NAMES = ["DoS", "Normal", "Probe", "R2L", "U2R"]
NUM_CLASSES = len(CLASS_NAMES)

# ─── Sequence parameters ──────────────────────────────────────────────────────
SEQUENCE_LENGTH = 10

# ─── Model architecture ───────────────────────────────────────────────────────
CNN_FILTERS_1     = 64
CNN_KERNEL_SIZE_1 = 3
CNN_FILTERS_2     = 128
CNN_KERNEL_SIZE_2 = 3
POOL_SIZE         = 2
LSTM_UNITS_1      = 128
DENSE_UNITS       = 128
DROPOUT_RATE      = 0.2
L2_REGULARIZATION = 1e-5

# ─── Focal Loss ───────────────────────────────────────────────────────────────
FOCAL_LOSS_GAMMA = 2.0
FOCAL_LOSS_ALPHA = 1.0   # placeholder; overridden at runtime by sqrt class weights
LABEL_SMOOTHING  = 0.05  # ε: soft targets, prevents overconfident majority-class predictions

# ─── Training hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE              = 256   # ↑ from 128: smoother gradients for minority classes
EPOCHS                  = 60
LEARNING_RATE           = 0.001 # ↑ from 0.0005: faster warm-start; ReduceLROnPlateau decays it
EARLY_STOPPING_PATIENCE = 15   # ↑ from 12: more room to recover after LR reductions
REDUCE_LR_PATIENCE      = 6    # ↑ from 5: fewer premature LR drops
REDUCE_LR_FACTOR        = 0.5
VALIDATION_SPLIT        = 0.15

# ─── Output file paths ────────────────────────────────────────────────────────
MODEL_SAVE_PATH            = os.path.join(MODELS_DIR, "cnn_lstm_model.keras")
SCALER_SAVE_PATH           = os.path.join(MODELS_DIR, "scaler.joblib")
LABEL_ENCODERS_SAVE_PATH   = os.path.join(MODELS_DIR, "label_encoders.joblib")
CLASSIFICATION_REPORT_PATH = os.path.join(OUTPUTS_DIR, "classification_report.json")
PER_CLASS_METRICS_PATH     = os.path.join(OUTPUTS_DIR, "per_class_metrics.json")
CONFUSION_MATRIX_PATH      = os.path.join(OUTPUTS_DIR, "confusion_matrix.png")
TRAINING_CURVES_PATH       = os.path.join(OUTPUTS_DIR, "training_curves.png")
ROC_CURVES_PATH            = os.path.join(OUTPUTS_DIR, "roc_curves.png")
ATTACK_DISTRIBUTION_PATH   = os.path.join(OUTPUTS_DIR, "attack_distribution.png")
MODEL_METADATA_PATH        = os.path.join(MODELS_DIR, "model_metadata.json")

print(f"Batch size              : {BATCH_SIZE}")
print(f"Learning rate           : {LEARNING_RATE}")
print(f"Focal Loss gamma        : {FOCAL_LOSS_GAMMA}")
print(f"Label smoothing (ε)     : {LABEL_SMOOTHING}")
print(f"EarlyStopping patience  : {EARLY_STOPPING_PATIENCE} epochs")
print(f"ReduceLR patience       : {REDUCE_LR_PATIENCE} epochs")
print(f"Sequence length         : {SEQUENCE_LENGTH}")
print("Configuration initialized.")

## 3. Data Ingestion

Parses the UNSW-NB15 CSV files and maps granular attack categories to the 5-class schema.

In [ ]:
import pandas as pd

def map_attack_label(attack_cat) -> str:
    cat = str(attack_cat).strip()
    if not cat or cat.lower() in ("nan", "none", ""):
        return "Normal"
    return ATTACK_MAPPING.get(cat, "Normal")

def load_unsw(file_path: str) -> pd.DataFrame:
    """Load one UNSW-NB15 CSV, auto-move from Colab root if needed."""
    filename = os.path.basename(file_path)
    if not os.path.exists(file_path) and os.path.exists(filename):
        import shutil
        shutil.move(filename, file_path)
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"File not found: {file_path}\n"
            "Upload UNSW_NB15_training-set.csv and UNSW_NB15_testing-set.csv "
            "via the Colab file browser (left sidebar → Files icon)."
        )
    print(f"Loading: {file_path}")
    df = pd.read_csv(file_path, low_memory=False)
    df.columns = df.columns.str.strip().str.lower()

    if "attack_cat" in df.columns:
        df["attack_category"] = df["attack_cat"].apply(map_attack_label)
    elif "label" in df.columns:
        df["attack_category"] = df["label"].apply(lambda x: "Normal" if int(x) == 0 else "DoS")
    else:
        raise ValueError("CSV must have 'attack_cat' or 'label' column.")

    df = df.drop(columns=[c for c in ["id", "label", "attack_cat"] if c in df.columns], errors="ignore")
    df = df.fillna(0)
    print(f"  {len(df):,} records | class distribution:")
    print(df["attack_category"].value_counts().to_string())
    return df

def load_train_test() -> tuple:
    return load_unsw(TRAIN_FILE), load_unsw(TEST_FILE)

print("Data loader defined.")

## 4. Data Preprocessing

**Pipeline:** log-transform skewed features → encode categoricals → encode targets → StandardScaler (fit on training set only).

In [ ]:
import numpy as np
import joblib
from collections import Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler

class DataPreprocessor:
    def __init__(self):
        self.label_encoders = {}
        self.scaler         = StandardScaler()
        self.target_encoder = LabelEncoder()
        self._is_fitted     = False

    def fit_transform(self, train_df: pd.DataFrame, test_df: pd.DataFrame):
        print("\n" + "=" * 60)
        print("PREPROCESSING PIPELINE")
        print("=" * 60)
        train = train_df.copy()
        test  = test_df.copy()

        # 1. Log-transform skewed features
        for col in SKEWED_COLS:
            if col in train.columns:
                train[col] = np.log1p(train[col].values.astype(np.float32))
                test[col]  = np.log1p(test[col].values.astype(np.float32))

        # 2. Encode categorical features (fit on union of train+test values)
        for col in CATEGORICAL_FEATURES:
            if col not in train.columns:
                continue
            le = LabelEncoder()
            le.fit(pd.concat([train[col], test[col]]).astype(str).unique())
            train[col] = le.transform(train[col].astype(str))
            test[col]  = le.transform(test[col].astype(str))
            self.label_encoders[col] = le

        # 3. Encode target labels (alphabetical order → matches LabelEncoder default)
        self.target_encoder.fit(CLASS_NAMES)
        y_train = self.target_encoder.transform(train["attack_category"].values)
        y_test  = self.target_encoder.transform(test["attack_category"].values)

        # 4. Extract feature matrices
        drop_cols = ["label", "attack_category"]
        feat_cols = [c for c in train.columns if c not in drop_cols]
        X_train = train[feat_cols].values.astype(np.float32)
        X_test  = test[feat_cols].values.astype(np.float32)

        # 5. Fit scaler ONLY on training set (prevents data leakage)
        self.scaler.fit(X_train)
        X_train = self.scaler.transform(X_train)
        X_test  = self.scaler.transform(X_test)

        self._is_fitted = True

        print(f"Features   : {X_train.shape[1]}")
        print(f"Train flat : {X_train.shape[0]:,} samples")
        print(f"Test flat  : {X_test.shape[0]:,} samples")
        print("\nTraining class distribution (flat):")
        dist = Counter(y_train)
        for idx, name in enumerate(CLASS_NAMES):
            print(f"  {name:8s}: {dist.get(idx, 0):6d}")
        return X_train, y_train, X_test, y_test

    def save_transformers(self) -> None:
        if not self._is_fitted:
            raise RuntimeError("Preprocessor not fitted yet.")
        joblib.dump(self.scaler, SCALER_SAVE_PATH)
        joblib.dump(
            {
                "label_encoders":       self.label_encoders,
                "target_encoder":       self.target_encoder,
                "dataset":              "unsw-nb15",
                "categorical_features": CATEGORICAL_FEATURES,
                "skewed_cols":          SKEWED_COLS,
                "constant_cols":        [],
            },
            LABEL_ENCODERS_SAVE_PATH,
        )
        print(f"Scaler saved   → {SCALER_SAVE_PATH}")
        print(f"Encoders saved → {LABEL_ENCODERS_SAVE_PATH}")

print("DataPreprocessor class defined.")

## 5. Sliding-Window Sequence Construction

Groups sequential flat records into temporal sliding-window sequences of length 10.

In [ ]:
from tensorflow.keras.utils import to_categorical

def build_sequences(X: np.ndarray, y: np.ndarray, seq_len: int = SEQUENCE_LENGTH) -> tuple:
    n, f = X.shape
    if n < seq_len:
        raise ValueError(f"Too few samples ({n}) for sequence length {seq_len}")
    n_seq = n - seq_len + 1
    print(f"Building sequences: {n:,} records → {n_seq:,} sequences (len={seq_len})")
    X_seq     = np.empty((n_seq, seq_len, f), dtype=np.float32)
    y_seq_raw = np.empty(n_seq, dtype=y.dtype)
    for i in range(n_seq):
        X_seq[i]     = X[i : i + seq_len]
        y_seq_raw[i] = y[i + seq_len - 1]
    y_seq = to_categorical(y_seq_raw, num_classes=NUM_CLASSES)
    print(f"  X_seq: {X_seq.shape}  |  y_seq: {y_seq.shape}")
    return X_seq, y_seq

print("Sequence builder defined.")

## 6. Loss, Metric & Model Architecture

**FocalLoss** with per-class alpha (square-root frequency weights) and label smoothing.  
**MacroF1Score** as EarlyStopping monitor — equal weight to all 5 classes.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# ─────────────────────────────────────────────────────────────────────────────
# Focal Loss with per-class alpha + label smoothing
# ─────────────────────────────────────────────────────────────────────────────
class FocalLoss(tf.keras.losses.Loss):
    """
    Multi-class focal loss with per-class alpha weighting and label smoothing.

    alpha: scalar float OR list of per-class weights (len = num_classes).
           When a list is passed (square-root frequency weights), the true-class
           weight is selected per sample via one-hot multiplication.
    label_smoothing (ε): replaces hard one-hot targets with soft targets:
           y_smooth = y_true*(1-ε) + ε/K
    """
    def __init__(self, gamma: float = 2.0, alpha=1.0, label_smoothing: float = 0.05, **kwargs):
        super().__init__(**kwargs)
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        # Normalize alpha to a plain Python type for JSON serialization
        if hasattr(alpha, "tolist"):
            alpha = alpha.tolist()
        elif hasattr(alpha, "numpy"):
            alpha = alpha.numpy().tolist()
        elif isinstance(alpha, (list, tuple)):
            alpha = [float(x) for x in alpha]
        else:
            alpha = float(alpha)
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        # Label smoothing: spread probability mass from true class to all classes
        if self.label_smoothing > 0.0:
            num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)

        cross_entropy = -y_true * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)

        # Per-class alpha: select the class weight of the true class per sample
        if isinstance(self.alpha, list):
            alpha_t = tf.reduce_sum(y_true * self.alpha, axis=-1, keepdims=True)
        else:
            alpha_t = self.alpha

        focal_weight = alpha_t * tf.pow(1.0 - p_t, self.gamma)
        focal_loss   = focal_weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(focal_loss, axis=-1))

    def get_config(self):
        return {**super().get_config(), "gamma": self.gamma, "alpha": self.alpha, "label_smoothing": self.label_smoothing}

# ─────────────────────────────────────────────────────────────────────────────
# Macro F1 Score (custom Keras metric)
# ─────────────────────────────────────────────────────────────────────────────
class MacroF1Score(tf.keras.metrics.Metric):
    def __init__(self, num_classes: int, name: str = "macro_f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.confusion_matrix = self.add_weight(
            name="confusion_matrix", shape=(num_classes, num_classes),
            initializer="zeros", dtype=tf.float32,
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_idx = tf.cast(tf.argmax(y_true, axis=-1), tf.int32)
        y_pred_idx = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
        self.confusion_matrix.assign_add(
            tf.math.confusion_matrix(y_true_idx, y_pred_idx,
                                     num_classes=self.num_classes, dtype=tf.float32)
        )

    def result(self):
        cm = self.confusion_matrix
        tp = tf.linalg.diag_part(cm)
        fp = tf.reduce_sum(cm, axis=0) - tp
        fn = tf.reduce_sum(cm, axis=1) - tp
        precision    = tp / (tp + fp + 1e-7)
        recall       = tp / (tp + fn + 1e-7)
        f1_per_class = 2.0 * precision * recall / (precision + recall + 1e-7)
        return tf.reduce_mean(f1_per_class)

    def reset_state(self):
        self.confusion_matrix.assign(
            tf.zeros((self.num_classes, self.num_classes), dtype=tf.float32)
        )

# ─────────────────────────────────────────────────────────────────────────────
# Bahdanau-style Attention Layer
# ─────────────────────────────────────────────────────────────────────────────
class AttentionLayer(layers.Layer):
    def build(self, input_shape):
        d = input_shape[-1]
        self.W = self.add_weight(name="W", shape=(d, d), initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(name="b", shape=(d,),   initializer="zeros",          trainable=True)
        self.v = self.add_weight(name="v", shape=(d, 1), initializer="glorot_uniform", trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        score   = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=[[2],[0]]) + self.b)
        weights = tf.nn.softmax(tf.tensordot(score, self.v, axes=[[2],[0]]), axis=1)
        context = tf.reduce_sum(inputs * weights, axis=1)
        return context, weights

    def get_config(self):
        return super().get_config()

# ─────────────────────────────────────────────────────────────────────────────
# Model builder
# ─────────────────────────────────────────────────────────────────────────────
def build_model(input_shape: tuple, num_classes: int = NUM_CLASSES, class_weights: list = None) -> Model:
    inputs = layers.Input(shape=input_shape, name="input_sequence")
    l2_reg = tf.keras.regularizers.l2(L2_REGULARIZATION)

    # CNN block — spatial feature extraction
    x = layers.Conv1D(CNN_FILTERS_1, CNN_KERNEL_SIZE_1, padding="same",
                      kernel_regularizer=l2_reg, name="conv1d_1")(inputs)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Activation("relu", name="relu_1")(x)
    x = layers.SpatialDropout1D(0.2, name="spatial_dropout_1")(x)

    x = layers.Conv1D(CNN_FILTERS_2, CNN_KERNEL_SIZE_2, padding="same",
                      kernel_regularizer=l2_reg, name="conv1d_2")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Activation("relu", name="relu_2")(x)
    x = layers.MaxPooling1D(pool_size=POOL_SIZE, name="maxpool")(x)
    x = layers.SpatialDropout1D(0.2, name="spatial_dropout_2")(x)

    # BiLSTM block — temporal pattern capture
    x = layers.Bidirectional(
        layers.LSTM(LSTM_UNITS_1, return_sequences=True, name="lstm_1"), name="bilstm"
    )(x)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_lstm")(x)

    # Attention block — focus on attack-indicative time steps
    context, _ = AttentionLayer(name="attention")(x)

    # Classification head
    x = layers.Dense(DENSE_UNITS, activation="relu",
                     kernel_regularizer=l2_reg, name="dense_1")(context)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_dense")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="output")(x)

    model = Model(inputs=inputs, outputs=outputs, name="CNN_LSTM_Attention_IDS")

    alpha = FOCAL_LOSS_ALPHA if class_weights is None else class_weights
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=FocalLoss(gamma=FOCAL_LOSS_GAMMA, alpha=alpha,
                       label_smoothing=LABEL_SMOOTHING, name="focal_loss"),
        metrics=["accuracy", MacroF1Score(num_classes=num_classes, name="macro_f1")],
    )
    model.summary()
    return model

print("Model components (FocalLoss + MacroF1 + AttentionLayer + build_model) defined.")

## 7. Training Callbacks & Functions

In [ ]:
import json
from datetime import datetime

def get_callbacks() -> list:
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_macro_f1", mode="max",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=MODEL_SAVE_PATH,
            monitor="val_macro_f1", mode="max",
            save_best_only=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_macro_f1", mode="max",
            factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE,
            min_lr=1e-6, verbose=1,
        ),
    ]

def train_model(model, X_train, y_train, X_val, y_val, sqr_weights) -> dict:
    print("\n" + "=" * 60)
    print("STARTING MODEL TRAINING")
    print("=" * 60)
    start = datetime.now()

    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        callbacks=get_callbacks(), verbose=1,
    )

    training_time = (datetime.now() - start).total_seconds()
    total_epochs  = len(history.history["loss"])
    print(f"\nTraining completed in {training_time:.1f}s ({total_epochs} epochs)")

    metadata = {
        "model_name":             "CNN_LSTM_Attention_IDS",
        "trained_at":             datetime.now().isoformat(),
        "training_time_seconds":  training_time,
        "total_epochs_trained":   total_epochs,
        "input_shape":            list(X_train.shape[1:]),
        "num_classes":            NUM_CLASSES,
        "class_names":            CLASS_NAMES,
        "sequence_length":        SEQUENCE_LENGTH,
        "dataset_used":           "unsw-nb15",
        "loss_function":          "FocalLoss",
        "focal_gamma":            FOCAL_LOSS_GAMMA,
        "focal_alpha":            sqr_weights,
        "label_smoothing":        LABEL_SMOOTHING,
        "batch_size":             BATCH_SIZE,
        "learning_rate":          LEARNING_RATE,
        "early_stopping_monitor": "val_macro_f1",
    }
    with open(MODEL_METADATA_PATH, "w") as f:
        json.dump(metadata, f, indent=2)

    history_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    history_data["training_time_seconds"] = training_time
    history_data["total_epochs"]          = total_epochs
    return history_data

print("Training functions defined.")

## 8. Evaluation & Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score, f1_score,
)

def evaluate_model(model, X_test: np.ndarray, y_test: np.ndarray) -> dict:
    print("\n" + "=" * 60)
    print("EVALUATING MODEL")
    print("=" * 60)

    y_pred_proba = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_test, axis=1)

    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    overall_acc      = accuracy_score(y_true, y_pred)
    overall_f1       = f1_score(y_true, y_pred, average="weighted")
    overall_macro_f1 = f1_score(y_true, y_pred, average="macro")

    print(f"Overall Accuracy  : {overall_acc:.4f}")
    print(f"Weighted F1 Score : {overall_f1:.4f}")
    print(f"Macro F1 Score    : {overall_macro_f1:.4f}")

    report["overall_accuracy"] = overall_acc
    report["weighted_f1"]      = overall_f1
    report["macro_f1"]         = overall_macro_f1

    with open(CLASSIFICATION_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2, default=str)

    # ROC AUC per class
    roc_auc_per_class = {}
    for i, cls_name in enumerate(CLASS_NAMES):
        if y_test[:, i].sum() > 0:
            fpr, tpr, _ = roc_curve(y_test[:, i], y_pred_proba[:, i])
            roc_auc_per_class[cls_name] = float(auc(fpr, tpr))
        else:
            roc_auc_per_class[cls_name] = None

    classes_payload = []
    for cls_name in CLASS_NAMES:
        m = report.get(cls_name, {})
        classes_payload.append({
            "name":      cls_name,
            "precision": round(float(m.get("precision", 0.0)), 4),
            "recall":    round(float(m.get("recall",    0.0)), 4),
            "f1":        round(float(m.get("f1-score",  0.0)), 4),
            "support":   int(m.get("support", 0)),
            "roc_auc":   round(roc_auc_per_class.get(cls_name) or 0.0, 4),
        })
    per_class = {
        "overall":     {"accuracy": round(overall_acc, 4), "weighted_f1": round(overall_f1, 4), "macro_f1": round(overall_macro_f1, 4)},
        "classes":     classes_payload,
        "class_names": CLASS_NAMES,
        "dataset":     "unsw-nb15",
    }
    with open(PER_CLASS_METRICS_PATH, "w") as f:
        json.dump(per_class, f, indent=2)

    # ── Confusion Matrix ──────────────────────────────────────────────────────
    cm            = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype("float") / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_title("Confusion Matrix (Counts)", fontsize=14)
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
    sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
    axes[1].set_title("Confusion Matrix (Normalized)", fontsize=14)
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
    plt.tight_layout()
    plt.savefig(CONFUSION_MATRIX_PATH, dpi=150, bbox_inches="tight")
    plt.show()
    with open(CONFUSION_MATRIX_PATH.replace(".png", ".json"), "w") as f:
        json.dump({"matrix": cm.tolist(), "normalized_matrix": cm_normalized.tolist(), "labels": CLASS_NAMES}, f, indent=2)

    # ── ROC Curves ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 8))
    roc_data = {}
    for i, cls_name in enumerate(CLASS_NAMES):
        if y_test[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_test[:, i], y_pred_proba[:, i])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2, label=f"{cls_name} (AUC={roc_auc:.3f})")
        roc_data[cls_name] = {"fpr": fpr.tolist(), "tpr": tpr.tolist(), "auc": float(roc_auc)}
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curves (One-vs-Rest)", fontsize=14)
    ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    plt.savefig(ROC_CURVES_PATH, dpi=150, bbox_inches="tight")
    plt.show()
    with open(ROC_CURVES_PATH.replace(".png", ".json"), "w") as f:
        json.dump(roc_data, f, indent=2)

    # ── Attack Distribution ───────────────────────────────────────────────────
    true_counts = np.bincount(y_true, minlength=NUM_CLASSES)
    pred_counts = np.bincount(y_pred, minlength=NUM_CLASSES)
    colors = ["#e74c3c", "#2ecc71", "#3498db", "#f39c12", "#9b59b6"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].bar(CLASS_NAMES, true_counts, color=colors)
    axes[0].set_title("Actual Attack Distribution", fontsize=14)
    axes[1].bar(CLASS_NAMES, pred_counts, color=colors)
    axes[1].set_title("Predicted Attack Distribution", fontsize=14)
    plt.tight_layout()
    plt.savefig(ATTACK_DISTRIBUTION_PATH, dpi=150, bbox_inches="tight")
    plt.show()
    with open(ATTACK_DISTRIBUTION_PATH.replace(".png", ".json"), "w") as f:
        json.dump({"labels": CLASS_NAMES, "actual_counts": true_counts.tolist(), "predicted_counts": pred_counts.tolist()}, f, indent=2)

    return report

def plot_training_curves(history: dict) -> None:
    epochs = range(1, len(history["loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, key, title, ylabel in [
        (axes[0], "loss",     "Focal Loss",     "Loss"),
        (axes[1], "accuracy", "Accuracy",       "Accuracy"),
        (axes[2], "macro_f1", "Macro F1 Score", "Macro F1"),
    ]:
        ax.plot(epochs, history[key], "b-", linewidth=2, label=f"Train")
        if f"val_{key}" in history:
            ax.plot(epochs, history[f"val_{key}"], "r-", linewidth=2, label="Val")
        ax.set_title(title, fontsize=13); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(TRAINING_CURVES_PATH, dpi=150, bbox_inches="tight")
    plt.show()

print("Evaluation and visualization functions defined.")

## 9. Run the Full Training Pipeline

**Steps:**
1. Load UNSW-NB15 CSV files
2. Preprocess (log-transform → encode → scale)
3. Build chronological sliding-window sequences
4. Stratified train/validation split (prevents homogeneous val set)
5. Compute square-root frequency class weights for Focal Loss alpha
6. Build and compile model
7. Train with EarlyStopping + ReduceLROnPlateau
8. Evaluate on test set
9. Plot training curves

In [ ]:
from sklearn.model_selection import train_test_split

# ── 1. Load ───────────────────────────────────────────────────────────────────
train_df, test_df = load_train_test()

# ── 2. Preprocess ─────────────────────────────────────────────────────────────
preprocessor = DataPreprocessor()
X_train_flat, y_train_flat, X_test_flat, y_test_flat = preprocessor.fit_transform(train_df, test_df)
preprocessor.save_transformers()

# ── 3. Build chronological sequences ──────────────────────────────────────────
X_train_seq_all, y_train_seq_all = build_sequences(X_train_flat, y_train_flat)
X_test_seq, y_test_seq           = build_sequences(X_test_flat,  y_test_flat)

# ── 4. Stratified train/validation split ──────────────────────────────────────
# Prevents the validation set from being homogeneous (all Normal traffic),
# which would lock val_macro_f1 near 0.2 and cause premature early stopping.
y_train_idx = np.argmax(y_train_seq_all, axis=1)
X_train_seq, X_val_seq, y_train_seq, y_val_seq = train_test_split(
    X_train_seq_all, y_train_seq_all,
    test_size=VALIDATION_SPLIT, random_state=42, stratify=y_train_idx,
)
print(f"Train sequences : {X_train_seq.shape[0]:,}")
print(f"Val sequences   : {X_val_seq.shape[0]:,}")
print(f"Test sequences  : {X_test_seq.shape[0]:,}")

# ── 5. Square-root frequency class weights (for Focal Loss alpha) ─────────────
# w_c = Σ√N_j / (C * √N_c)  — milder than inverse-frequency, avoids gradient explosion
unique_classes, counts = np.unique(y_train_flat, return_counts=True)
sqr_counts  = np.sqrt(counts)
sqr_weights = (np.sum(sqr_counts) / (NUM_CLASSES * sqr_counts)).tolist()
print("\nSquare-root class weights (Focal Loss alpha):")
for name, w in zip(CLASS_NAMES, sqr_weights):
    print(f"  {name:8s}: {w:.4f}")

# ── 6. Build model ────────────────────────────────────────────────────────────
input_shape = (X_train_seq.shape[1], X_train_seq.shape[2])
model = build_model(input_shape, class_weights=sqr_weights)

# ── 7. Train ──────────────────────────────────────────────────────────────────
history = train_model(model, X_train_seq, y_train_seq, X_val_seq, y_val_seq, sqr_weights)

# ── 8. Evaluate ───────────────────────────────────────────────────────────────
report = evaluate_model(model, X_test_seq, y_test_seq)

# ── 9. Training curves ────────────────────────────────────────────────────────
plot_training_curves(history)

print("\n" + "=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)
print(f"Overall Accuracy  : {report['overall_accuracy']:.4f}")
print(f"Weighted F1       : {report['weighted_f1']:.4f}")
print(f"Macro F1          : {report['macro_f1']:.4f}")

## 10. Download Trained Artifacts to Local Computer

Run this cell after training finishes. Place the downloaded files in the local project's `models/` and `outputs/` directories.

In [ ]:
import time

try:
    from google.colab import files

    files_to_download = [
        SCALER_SAVE_PATH,
        LABEL_ENCODERS_SAVE_PATH,
        MODEL_SAVE_PATH,
        MODEL_METADATA_PATH,
        CLASSIFICATION_REPORT_PATH,
        PER_CLASS_METRICS_PATH,
        CONFUSION_MATRIX_PATH,
        CONFUSION_MATRIX_PATH.replace(".png", ".json"),
        TRAINING_CURVES_PATH,
        ROC_CURVES_PATH,
        ROC_CURVES_PATH.replace(".png", ".json"),
        ATTACK_DISTRIBUTION_PATH,
        ATTACK_DISTRIBUTION_PATH.replace(".png", ".json"),
    ]

    print("Starting downloads...")
    for f_path in files_to_download:
        if os.path.exists(f_path):
            print(f"  ↓ {f_path}")
            files.download(f_path)
            time.sleep(0.8)
        else:
            print(f"  ✗ Not found (skipping): {f_path}")
    print("\nAll files downloaded. Place them in models/ and outputs/ in your local project.")
except ImportError:
    print("Not running in Google Colab. Files are saved locally in models/ and outputs/.")